In [1]:
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Load both models ──────────────────────────────────────
demo_model = joblib.load("models/demographic_classifier.pkl")

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("models/roberta_checkpoint")
roberta   = AutoModelForSequenceClassification.from_pretrained(
    "models/roberta_checkpoint"
)
roberta   = roberta.to(device)
roberta.eval()

In [ ]:
# ── Load test data ─────────────────────────────────────────
product_test = pd.read_csv("data/final/test.csv")
demo_test    = pd.read_csv("data/processed/demographic_features.csv")

In [ ]:
# ── Get RoBERTa probability scores ────────────────────────
def get_roberta_probs(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            batch, max_length=256, padding=True,
            truncation=True, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out   = roberta(**enc)
            probs = torch.softmax(out.logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs)

print("Getting RoBERTa probabilities...")
roberta_probs = get_roberta_probs(product_test['input_text'].tolist())
# Shape: (n_samples, 2) — col 0 = P(S2), col 1 = P(S1)